### **Earthquake_Event_Data-Gold_Layer**

In [0]:
%pip install reverse_geocoder

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/2.2 MB ? eta -:--:--
     ━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.1/2.2 MB 1.8 MB/s eta 0:00:02
     ━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.3/2.2 MB 4.3 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━ 0.9/2.2 MB 9.1 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 2.2/2.2 MB 17.5 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 14.4 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for reverse_geocoder: filename=reverse_geocoder-1.5.1-py3-none-any.whl size=2268066 sha256=98c2afb2f4ea9fdc4ffce98030d76ff6e69c3d4d582197188773ab428bfbd8b8
  Stored in directory: /home/spark-de57006b-bec7-4e6a-8718-e8/.cache/pip/wheels/17/3c/41/2bc89719586c2a5c53e9a527daa76a968a1288315c1ae2d904
Successfully built reverse_geocoder
Note: you may need to restart the kernel using %restart_python or dbutils.library.resta

In [0]:
spark.conf.set("fs.azure.account.auth.type.adlsweatherapidata.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.adlsweatherapidata.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.adlsweatherapidata.dfs.core.windows.net", "bf472d2f-5f2c-4d09-941f-7117dc63e911")
spark.conf.set("fs.azure.account.oauth2.client.secret.adlsweatherapidata.dfs.core.windows.net", "-xb8Q~WbkDR~RKt6JuL3qpKRclXrTDDrBcgurbsM")
spark.conf.set("fs.azure.account.oauth2.client.endpoint.adlsweatherapidata.dfs.core.windows.net", "https://login.microsoftonline.com/cc219978-6391-4153-8405-87dd0faa993b/oauth2/token")

In [0]:
from pyspark.sql.functions import when, col, udf
from pyspark.sql.types import StringType
# Ensure the below library is installed on environment
import reverse_geocoder as rg

In [0]:
df = spark.read.table("earthquake_silver")

In [0]:
def get_country_code(lat, lon):

    coordinates = (float(lat), float(lon))
    return rg.search(coordinates) [0].get('cc')

In [0]:
# Registering the udf so they can be used on spark dataframe
get_country_code_udf = udf(get_country_code, StringType())

In [0]:
df_with_location = \
                   df.\
                       withColumn("country_code", get_country_code_udf(col("Latitude"), col("Longitude")))

In [0]:
# adding significance classification 
df_with_location_sig_class = \
                              df_with_location.\
                                  withColumn('sig_class',
                                             when(col("sig") < 100, "Low").\
                                             when((col("sig") >= 100) & (col("sig") < 500), "moderte").\
                                             otherwise("High")
                                            )

In [0]:
df_with_location_sig_class.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("earthquake_gold")